# CS F407 - Artificial Intelligence
## Laboratory: Logical Reasoning for Planning

**Student:** Eshan Saikia  
**ID:** 2023A7PS1024G

This notebook contains the executable planning agent and validation evidence. The accompanying report contains the written specification, reasoning, LLM-use discussion, and reflections.

## 1. Planning Problem

The warehouse has three locations \(A,B,C\). The robot and package start at \(A\), and the goal is to have the package at \(C\).

\[
I=\{\mathrm{At(Robot,A)},\mathrm{At(Package,A)}\}
\]

\[
G=\{\mathrm{At(Package,C)}\}.
\]

The connected movement links are:

\[
A\leftrightarrow B,\qquad B\leftrightarrow C.
\]

### Action definitions

| Action | Preconditions | Negative effects | Positive effects |
|---|---|---|---|
| `Move(X,Y)` | `At(Robot,X)` | `At(Robot,X)` | `At(Robot,Y)` |
| `PickUp(Package,X)` | `At(Robot,X)`, `At(Package,X)` | `At(Package,X)` | `Holding(Package)` |
| `Drop(Package,X)` | `At(Robot,X)`, `Holding(Package)` | `Holding(Package)` | `At(Package,X)` |

An action is applicable exactly when all its positive preconditions are present and all its negative preconditions (if any) are absent from the current state.

From the initial state, the applicable actions are `Move(A,B)` and `PickUp(Package,A)`: both have their preconditions satisfied. `Drop(Package,C)` is not applicable because the robot is not at `C` and is not holding the package.

### Manual plan

A valid plan is:

\[
\mathrm{PickUp(Package,A)}
\rightarrow
\mathrm{Move(A,B)}
\rightarrow
\mathrm{Move(B,C)}
\rightarrow
\mathrm{Drop(Package,C)}.
\]

The corresponding states are:

| State | Facts |
|---|---|
| \(S_0\) | `At(Robot,A)`, `At(Package,A)` |
| \(S_1\) | `At(Robot,A)`, `Holding(Package)` |
| \(S_2\) | `At(Robot,B)`, `Holding(Package)` |
| \(S_3\) | `At(Robot,C)`, `Holding(Package)` |
| \(S_4\) | `At(Robot,C)`, `At(Package,C)` |

## 2. LLM-Assisted Planner Design

The implementation was requested from the LLM using the following specification.

> I want to implement a simple planning agent in Python. Represent a state as a set of logical propositions. Each action should contain a name; positive preconditions; negative preconditions; positive effects; and negative effects. An action is applicable if all positive preconditions are satisfied and all negative preconditions are absent. When an action is applied, remove its negative effects from the state and add its positive effects. Use breadth-first search to find an action sequence that achieves a specified goal. The program should detect when no plan exists, print the resulting action sequence, and print the states reached after each action. Use the three-location warehouse with movement A-B and B-C, the package initially at A, and the goal package-at-C. Keep it dependency-free and easy to inspect.

The generated planner structure was accepted after inspection. I then added explicit test cases, plan validation, a reusable state-transition trace, and assertions for the required behaviours.

In [1]:
from dataclasses import dataclass
from collections import deque
from typing import FrozenSet, Iterable, Optional

@dataclass(frozen=True)
class Action:
    name: str
    positive_preconditions: FrozenSet[str]
    negative_preconditions: FrozenSet[str]
    positive_effects: FrozenSet[str]
    negative_effects: FrozenSet[str]

    def applicable(self, state: FrozenSet[str]) -> bool:
        return (
            self.positive_preconditions <= state
            and self.negative_preconditions.isdisjoint(state)
        )

    def apply(self, state: FrozenSet[str]) -> FrozenSet[str]:
        if not self.applicable(state):
            raise ValueError(f"Action is not applicable: {self.name}")
        next_state = (state - self.negative_effects) | self.positive_effects
        return frozenset(next_state)

def plan_bfs(
    initial_state: FrozenSet[str],
    actions: Iterable[Action],
    goal: FrozenSet[str],
):
    actions = tuple(actions)
    frontier = deque([(initial_state, [])])
    visited = {initial_state}
    expanded = 0

    while frontier:
        state, plan = frontier.popleft()
        expanded += 1

        if goal <= state:
            return plan, state, expanded

        for action in actions:
            if action.applicable(state):
                successor = action.apply(state)
                if successor not in visited:
                    visited.add(successor)
                    frontier.append((successor, plan + [action]))

    return None, None, expanded

def fact(robot_location, package_location=None, holding=False):
    state = {f"At(Robot,{robot_location})"}
    if holding:
        state.add("Holding(Package)")
    elif package_location is not None:
        state.add(f"At(Package,{package_location})")
    return frozenset(state)

def move(x, y):
    return Action(
        name=f"Move({x},{y})",
        positive_preconditions=frozenset({f"At(Robot,{x})"}),
        negative_preconditions=frozenset(),
        positive_effects=frozenset({f"At(Robot,{y})"}),
        negative_effects=frozenset({f"At(Robot,{x})"}),
    )

def pickup(location):
    return Action(
        name=f"PickUp(Package,{location})",
        positive_preconditions=frozenset({
            f"At(Robot,{location})",
            f"At(Package,{location})",
        }),
        negative_preconditions=frozenset(),
        positive_effects=frozenset({"Holding(Package)"}),
        negative_effects=frozenset({f"At(Package,{location})"}),
    )

def drop(location):
    return Action(
        name=f"Drop(Package,{location})",
        positive_preconditions=frozenset({
            f"At(Robot,{location})",
            "Holding(Package)",
        }),
        negative_preconditions=frozenset(),
        positive_effects=frozenset({f"At(Package,{location})"}),
        negative_effects=frozenset({"Holding(Package)"}),
    )

MOVE_ACTIONS = [
    move("A", "B"),
    move("B", "A"),
    move("B", "C"),
    move("C", "B"),
]

ALL_ACTIONS = (
    *MOVE_ACTIONS,
    pickup("A"),
    pickup("B"),
    pickup("C"),
    drop("A"),
    drop("B"),
    drop("C"),
)

INITIAL = fact("A", package_location="A")
GOAL = frozenset({"At(Package,C)"})

print("Initial state:", sorted(INITIAL))
print("Goal:", sorted(GOAL))

Initial state: ['At(Package,A)', 'At(Robot,A)']
Goal: ['At(Package,C)']


## 3. Solvable Planning Problem

The required planner should find a sequence that transports the package from \(A\) to \(C\). Each returned action is checked for applicability before its state transition is accepted.

In [2]:
plan, final_state, expanded = plan_bfs(INITIAL, ALL_ACTIONS, GOAL)

print("Plan found:", plan is not None)
print("States expanded:", expanded)

for i, action in enumerate(plan or [], start=1):
    print(f"{i}. {action.name}")

def validate_plan(initial_state, plan, goal):
    state = initial_state
    trace = [state]

    for action in plan:
        assert action.applicable(state), f"Invalid action: {action.name}"
        state = action.apply(state)
        trace.append(state)

    assert goal <= state
    return trace

trace = validate_plan(INITIAL, plan, GOAL)

print("\nState trace:")
for i, state in enumerate(trace):
    print(f"S{i}:", sorted(state))

print("\nPlan independently validated:", GOAL <= trace[-1])

Plan found: True
States expanded: 8
1. PickUp(Package,A)
2. Move(A,B)
3. Move(B,C)
4. Drop(Package,C)

State trace:
S0: ['At(Package,A)', 'At(Robot,A)']
S1: ['At(Robot,A)', 'Holding(Package)']
S2: ['At(Robot,B)', 'Holding(Package)']
S3: ['At(Robot,C)', 'Holding(Package)']
S4: ['At(Package,C)', 'At(Robot,C)']

Plan independently validated: True


## 4. Impossible Problem

Test B removes the pickup actions. The robot can still move through the warehouse, but the package cannot be placed into the robot's `Holding(Package)` state. Therefore no sequence can satisfy `At(Package,C)`.

In [3]:
NO_PICKUP_ACTIONS = tuple(MOVE_ACTIONS) + (
    drop("A"), drop("B"), drop("C"),
)

impossible_plan, _, impossible_expanded = plan_bfs(
    INITIAL,
    NO_PICKUP_ACTIONS,
    GOAL,
)

print("Plan found:", impossible_plan is not None)
print("States expanded:", impossible_expanded)

assert impossible_plan is None
print("Correctly reports: No plan found")

Plan found: False
States expanded: 3
Correctly reports: No plan found


## 5. Irrelevant Robot-Movement Actions

The planner must not confuse the robot reaching \(C\) with the package reaching \(C\). This test allows the robot to move freely but provides no pickup/drop capability. Even if the robot reaches `C`, the goal fact `At(Package,C)` is absent.

In [4]:
ROBOT_ONLY_ACTIONS = (
    move("A", "B"),
    move("B", "A"),
    move("B", "C"),
    move("C", "B"),
)

robot_only_plan, robot_only_final, robot_only_expanded = plan_bfs(
    INITIAL,
    ROBOT_ONLY_ACTIONS,
    GOAL,
)

print("Plan found:", robot_only_plan is not None)
print("States expanded:", robot_only_expanded)

assert robot_only_plan is None
print("Correctly does not treat robot-at-C as package-at-C.")

Plan found: False
States expanded: 3
Correctly does not treat robot-at-C as package-at-C.


## 6. Logic and Search

For a current state \(S\) and action \(a\):

\[
S \models \mathrm{Preconditions}(a)
\]

is the logical applicability check. If the check succeeds, the action produces

\[
S'=\mathrm{Apply}(S,a).
\]

The planner then uses BFS to decide which applicable successor states to explore. Thus:

```text
Current state
      ↓
Check action preconditions
      ↓
Applicable actions
      ↓
Generate successor state
      ↓
Search over alternatives
      ↓
Goal?
```

Logic determines which transitions are legal; search determines which legal sequences are explored.

## 7. Think About It

**Task 0 — applicability:** An action is not applicable merely because it is listed; every required precondition must hold in the current state. This is why `PickUp(Package,A)` is applicable initially while `Drop(Package,C)` is not.

**Task 1 — design before prompting:** Writing the state, actions, transitions, goal, and required outputs first gives a fixed specification for evaluating the LLM-generated implementation.

**Task 2 — mapping the specification to code:** Preconditions are checked in `Action.applicable()`, effects are applied in `Action.apply()`, the goal is checked in `plan_bfs()`, and BFS is implemented by the FIFO `deque` frontier over applicable successor states.

**Task 4 — logic and search:** Logic determines which actions are legal from a state; search determines which legal action sequences are explored.

The optional LLM self-verification task is not required for the main submission.

## 8. Final Verification

The implementation was checked against the laboratory requirements and the expected behaviours:

- applicability depends on explicit preconditions;
- negative effects are removed and positive effects are added;
- BFS explores alternative applicable action sequences;
- goal testing checks the package location, not merely the robot location;
- no-plan cases terminate without inventing actions;
- every action in the returned plan is independently re-checked before accepting the plan;
- the executable results are recorded above.

The optional Prolog extension in the laboratory specification is not included because it is explicitly marked **optional**.

## 9. Reflection Questions

1. **Why specify preconditions and effects before asking an LLM to write the planner?**  
   Preconditions and effects define the semantics of each action independently of the implementation. They provide a precise specification for checking whether generated code represents the intended planning problem.

2. **What error could occur if preconditions were not checked?**  
   The planner could execute `PickUp(Package,B)` while the robot and package are at different locations, creating a state that is not reachable under the intended action semantics.

3. **Why is a plan that looks reasonable not necessarily valid?**  
   Every action must be applicable in the state where it occurs, and the final state must satisfy the formal goal. A sequence can therefore look sensible while containing an illegal transition or ending in the wrong state.

4. **What did the LLM contribute?**  
   It translated the specified state/action representation and BFS strategy into an executable Python starting point, reducing implementation effort.

5. **What had to be verified independently?**  
   The applicability test, effect application, goal test, BFS exploration, failure behaviour, and every action/state transition in the returned plan were independently checked.

6. **Where is logical reasoning being used?**  
   Logic is used when checking whether a state's facts satisfy an action's preconditions and when applying the action's stated positive and negative effects.

7. **How is planning related to the search algorithms studied previously?**  
   Planning searches over states generated by actions, just as ordinary search explores successor states. The difference is that planning defines those successors through logical action preconditions and effects.

## 9. Verification Checklist

- [x] Initial state, goal, and all action preconditions/effects.
- [x] Manual plan with every intermediate state.
- [x] Exact LLM implementation prompt.
- [x] Solvable, impossible, and irrelevant-action tests.
- [x] Independent validation of every returned plan action.
- [x] Logic-versus-search explanation.
- [x] All seven reflection questions.
- [x] Optional Prolog extension identified as optional and not required for this submission.